In [ ]:
import re
import json
import os
import numpy as np
import pandas as pd
import pickle
from typing import List, Dict, Any, Iterable, Union

Number = Union[int, np.integer]
NONE_VALUE = -1

def get_average_tokens(model_outputs: List[Dict]) -> List[float]:
    avg_tokens: List[float] = []
    for user_output in model_outputs:
        tokens = [out.get('num_tokens', 0) for out in user_output.get('outputs', [])]
        avg_token = float(np.mean(tokens)) if tokens else 0.0
        avg_tokens.append(avg_token)
    return avg_tokens


_OUTPUT_BLOCK_RE = re.compile(
    r"<output>\s*(?:Ranking|Recommendation)\s+result\s*:\s*(?:\[(?P<br>.*?)\]|(?P<nobr>[\d,\s，]+))\s*</output>",
    flags=re.IGNORECASE | re.DOTALL
)
_OUTPUT_INLINE_RE = re.compile(
    r"(?:Ranking|Recommendation)\s+result\s*:\s*(?:\[(?P<br>.*?)\]|(?P<nobr>[\d,\s，]+))",
    flags=re.IGNORECASE | re.DOTALL
)

def _extract_id_list(text: str) -> List[int]:
    # 1) <output> ... </output>
    m = _OUTPUT_BLOCK_RE.search(text or "")
    if not m:
        m = _OUTPUT_INLINE_RE.search(text or "")
    if m:
        inside = m.group("br") if m.group("br") is not None else m.group("nobr")
        return [int(x) for x in re.findall(r"\d+", inside or "")]

    if re.fullmatch(r"\s*\d+(?:\s*,\s*\d+)*\s*", text or ""):
        return [int(x) for x in re.findall(r"\d+", text)]

    m = re.search(r"\[(.*?)\]", text or "", flags=re.DOTALL)
    if m:
        return [int(x) for x in re.findall(r"\d+", m.group(1))]
    return []


def parse_predictions(model_outputs: List[Dict]) -> List[List[List[int]]]:
    all_preds: List[List[List[int]]] = []
    for user_output in model_outputs:
        user_preds: List[List[int]] = []
        for out in user_output.get('outputs', []):
            text = (out.get('text') or '').strip()
            ids = _extract_id_list(text)
            user_preds.append(ids)
        all_preds.append(user_preds)
    return all_preds


def _to_gt_set(gt: Union[Number, Iterable[Number]]) -> set:
    if isinstance(gt, (int, np.integer)):
        return {int(gt)}
    return {int(x) for x in gt}

def _dedup_keep_order(seq):
    seen = set()
    out = []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def compute_ndcg_at_k(pred: List[int], ground_truth, k: int = 10) -> float:
    if not pred:
        return NONE_VALUE
    gt = _to_gt_set(ground_truth)
    if not gt:
        return 0.0

    pred_clean = _dedup_keep_order(pred)[:max(1, int(k))]

    # DCG
    dcg = 0.0
    for i, pid in enumerate(pred_clean):
        if pid in gt:
            dcg += 1.0 / np.log2(i + 2.0)

    # IDCG
    m = min(len(gt), len(pred_clean))  
    idcg = sum(1.0 / np.log2(i + 2.0) for i in range(m))
    if idcg == 0.0:
        return 0.0

    ndcg = dcg / idcg
    return float(min(1.0, max(0.0, ndcg)))


def compute_recall_at_k(pred: List[int], ground_truth, k: int = 10) -> float:
  
    if not pred:
        return NONE_VALUE
    gt = _to_gt_set(ground_truth)
    if not gt:
        return 0.0

    pred_clean = _dedup_keep_order(pred)[:max(1, int(k))]
    hit = len(set(pred_clean) & gt)
    return float(hit) / float(len(gt))



def evaluate_all_minimal(
    model_outputs: List[Dict],
    ground_truths: List[Union[Number, Iterable[Number]]],
) -> pd.DataFrame:
    parsed_preds = parse_predictions(model_outputs)

    records = []
    for idx, (user_preds, gt) in enumerate(zip(parsed_preds, ground_truths)):
        rec10_list = []
        ndcg10_list = []
        for pred in user_preds[:5]:  
            rec10_list.append(compute_recall_at_k(pred, gt, k=10))
            ndcg10_list.append(compute_ndcg_at_k(pred, gt, k=10))

        while len(rec10_list) < 5:
            rec10_list.append(NONE_VALUE)
        while len(ndcg10_list) < 5:
            ndcg10_list.append(NONE_VALUE)

        rec5_vals  = [compute_recall_at_k(pred, gt, k=5)  for pred in user_preds]
        ndcg5_vals = [compute_ndcg_at_k(pred, gt, k=5)    for pred in user_preds]
        rec5_valid  = [v for v in rec5_vals  if v != -1.0]
        ndcg5_valid = [v for v in ndcg5_vals if v != -1.0]
        avg_recall_5 = float(np.mean(rec5_valid))  if rec5_valid  else 0.0
        avg_ndcg_5   = float(np.mean(ndcg5_valid)) if ndcg5_valid else 0.0

        rec10_valid  = [v for v in rec10_list  if v != -1.0]
        ndcg10_valid = [v for v in ndcg10_list if v != -1.0]
        avg_recall_10 = float(np.mean(rec10_valid))  if rec10_valid  else 0.0
        avg_ndcg_10   = float(np.mean(ndcg10_valid)) if ndcg10_valid else 0.0

        record = {"index": idx}
        for i in range(5):
            record[f"recall10_{i+1}"] = rec10_list[i]
            record[f"ndcg10_{i+1}"]   = ndcg10_list[i]
        record["avg_recall@5"] = avg_recall_5
        record["avg_ndcg@5"]   = avg_ndcg_5
        record["avg_recall@10"] = avg_recall_10
        record["avg_ndcg@10"]   = avg_ndcg_10

        records.append(record)

    cols_order = (
        ["index"]
        + [f"recall10_{i}" for i in range(1, 6)]
        + [f"ndcg10_{i}"   for i in range(1, 6)]
        + ["avg_recall@5", "avg_ndcg@5", "avg_recall@10", "avg_ndcg@10"]
    )
    df = pd.DataFrame(records)
    return df[cols_order]


def get_results(system, eval_data, dataset_name, mode, saved_file_path):

    output_file = f'{system}_{dataset_name}_{mode}_llm_output.jsonl'
    with open(os.path.join(saved_file_path, output_file), "r", encoding="utf-8") as f:
        data = json.load(f)

    averages = get_average_tokens(data)

    gts = [eval_data[i][1] for i in range(len(eval_data))]

    results = evaluate_all_minimal(data, gts)
    return data, averages, results



def get_mean_ndcg(results: pd.DataFrame, k: int = 10) -> float:
    col = f'avg_ndcg@{k}'
    if col in results.columns:
        return float(results[col].mean())
    raise KeyError(f"NDCG column not found for k={k}")


def get_mean_recall(results: pd.DataFrame, k: int = 10) -> float:

    col = f'avg_recall@{k}'
    if col in results.columns:
        return float(results[col].mean())
    raise KeyError(f"Recall column not found for k={k}")



def get_label(cot_results: pd.DataFrame, direct_results: pd.DataFrame, k: int = 10):
    col = f'avg_ndcg@{k}' if f'avg_ndcg@{k}' in cot_results.columns else 'avg_ndcg'
    cot_avg_ndcg = cot_results[col].tolist()
    direct_avg_ndcg = direct_results[col].tolist()

    label = []
    for i in range(len(cot_avg_ndcg)):
        if cot_avg_ndcg[i] > direct_avg_ndcg[i]:
            label.append(1)
        elif cot_avg_ndcg[i] < direct_avg_ndcg[i]:
            label.append(0)
        else:
            label.append(2)
    return label


In [ ]:
def get_csv_results(system, dataset_name, model, mode, K_list, version = 'ranking_v4', saved_flag = 0):
    seed = 2025
    PATH = os.path.join('/data/', f'{dataset_name}/{model}')
    data_path = os.path.join('/data/', f'{dataset_name}/processed_data')
    saved_file_path = os.path.join(PATH, 'saved_data')
    saved_result_path = os.path.join(PATH, 'saved_result')
    if not os.path.exists(saved_file_path):
        os.makedirs(saved_file_path)
    if not os.path.exists(saved_result_path):
        os.makedirs(saved_result_path)

    with open(os.path.join(data_path, f'{dataset_name}_train.txt'), "rb") as file:
        train_data = pickle.load(file)
    train_data = list(train_data)
    with open(os.path.join(data_path, f'{dataset_name}_test.txt'), "rb") as file:
        test_data = pickle.load(file)
    test_data = list(test_data)

    if mode == 'train':
        eval_data = train_data
    else:
        eval_data = test_data

    name = (system or "").lower()
    result_list = []
    data, averages, results = get_results(system, eval_data, dataset_name, mode, saved_file_path)
    preds = parse_predictions(data)
    if saved_flag:
        results.to_csv(os.path.join(saved_result_path, f'{system}_{mode}_ndcg.csv'), index=False)
        np.save(os.path.join(saved_result_path, f'{system}_{mode}_averages.npy'), averages)
    for k in K_list:
        result_list.append(get_mean_recall(results, k=k))
        result_list.append(get_mean_ndcg(results, k=k))
    result_list.append(np.mean(averages))

    return data, averages, results, result_list

In [3]:
dataset_name = 'ml-1m'
result_csv = []
saved_flag = 1
for model in ['Qwen3-14B']:
    # for system in ['direct', 'CoT']:
    for system in ['ranking_direct_v4', 'ranking_CoT_v4']:
        for mode in ['test']:
            K_list = [5, 10]
            version = 'ranking_v4'
            data, averages, results, result_list = get_csv_results(system, dataset_name, model, mode, K_list, version, saved_flag)
            result_csv.append(result_list)
result_csv = pd.DataFrame(result_csv, columns=['recall@5', 'ndcg@5', 'recall@10', 'ndcg@10', 'avg_tokens'])

In [14]:
result_csv.to_csv(f'./results/csv/{dataset_name}_{mode}.csv')

In [ ]:
def get_mixed_results(cot_results, direct_results, cot_tokens, direct_tokens):
    cot_avg_ndcg = cot_results['avg_ndcg@10'].tolist()
    direct_avg_ndcg = direct_results['avg_ndcg@10'].tolist()
    mixed_avg_ndcg = []
    mixed_tokens = []
    count_i = []
    for i in range(len(cot_avg_ndcg)):
        if cot_avg_ndcg[i] > direct_avg_ndcg[i]:
            mixed_avg_ndcg.append(cot_avg_ndcg[i])
            mixed_tokens.append(cot_tokens[i])
            count_i.append(i)
        else:
            mixed_avg_ndcg.append(direct_avg_ndcg[i])
            mixed_tokens.append(direct_tokens[i])
    return mixed_avg_ndcg, mixed_tokens, count_i

dataset_name = 'ml-1m'
model = 'Qwen3-4B'
mode = 'test'
K_list = [10]
version = 'ranking_v4'
cot_system = 'ranking_CoT_v4'
direct_system = 'ranking_direct_v4'
cot_data, cot_averages, cot_results, cot_result_list = get_csv_results(cot_system, dataset_name, model, mode, K_list, version)
direct_data, direct_averages, direct_results, direct_result_list = get_csv_results(direct_system, dataset_name, model, mode, K_list, version)
mixed_avg_ndcg, mixed_tokens, count_i = get_mixed_results(cot_results, direct_results, cot_averages, direct_averages)
print(f'Mixed mean NDCG: {np.mean(mixed_avg_ndcg)}')
print(f'Mixed average tokens: {np.mean(mixed_tokens)}')

In [7]:
def compare_results(cot_results, direct_results):
    cot_avg_ndcg = cot_results['avg_ndcg@10'].tolist()
    direct_avg_ndcg = direct_results['avg_ndcg@10'].tolist()
    cot_count = 0
    direct_count = 0
    for i in range(len(cot_avg_ndcg)):
        if cot_avg_ndcg[i] > direct_avg_ndcg[i]:
            cot_count += 1
        elif cot_avg_ndcg[i] < direct_avg_ndcg[i]:
            direct_count += 1
    print(f'CoT count: {cot_count / len(cot_avg_ndcg)}, Direct count: {direct_count / len(direct_avg_ndcg)}')

compare_results(cot_results, direct_results)

CoT count: 0.37085843373493976, Direct count: 0.32379518072289154
